# EDA — Raw County Real Estate Datasets
Exploring the four raw sources under `data/raw/` to answer:
1. Which county does each dataset cover, and what do its records represent?
2. Do the datasets share columns/keys that could be used to merge them?
3. What is the main idea (purpose/grain) of each dataset?

In [1]:
import json
import pandas as pd
import matplotlib as plt
import pathlib as Path
from src.utils.load_config import load_config

cfg = load_config()

In [2]:
ROOT_PATH = cfg["paths"]["root"]
RAW_PATH = ROOT_PATH +"/"+ cfg["paths"]["raw_data"]

## 1. Load each raw source
- `Cama_Table_transformed.csv` — a semicolon-delimited CAMA (Computer Assisted Mass Appraisal) export, no county name in the path.
- `buncombe/buncombe.json` and `guilford/guilford.json` — the **full** result of paging an ArcGIS FeatureServer `query` endpoint (see `config/config.yaml` sources) with `src/components/data_ingestor.py`, which loops on `resultOffset`/`resultRecordCount` until every record is retrieved (verified against each service's `returnCountOnly` total).
- `wake/RealEstData09022026.xlsx` — a Wake County real-estate extract, single sheet `REALINFO`.

In [3]:
def load_arcgis_json(path):
    """Flatten an ArcGIS FeatureServer query response into a DataFrame.
    Keeps attributes as columns; geometry is kept as a raw dict in `_geometry`.
    """
    with open(path) as f:
        payload = json.load(f)
    df = pd.DataFrame([f["attributes"] for f in payload["features"]])
    df["_geometry"] = [f.get("geometry") for f in payload["features"]]
    meta = {
        "geometryType": payload.get("geometryType"),
        "exceededTransferLimit": payload.get("exceededTransferLimit"),
        "n_features": len(payload["features"]),
    }
    return df, meta

meck_df = pd.read_csv(RAW_PATH + "Cama_Table_transformed.csv", sep=";", encoding="utf-8-sig", decimal=",", low_memory=False)
bun_df, bun_meta = load_arcgis_json(RAW_PATH + "buncombe/buncombe.json")
gui_df, gui_meta = load_arcgis_json(RAW_PATH + "guilford/guilford.json")
wake_df = pd.read_excel(RAW_PATH + "wake/RealEstData09022026.xlsx", sheet_name="REALINFO")

print("Mecklenburg CAMA:", meck_df.shape)
print("Buncombe (full ArcGIS pull):", bun_df.shape, bun_meta)
print("Guilford (full ArcGIS pull):", gui_df.shape, gui_meta)
print("Wake REALINFO:", wake_df.shape)

Mecklenburg CAMA: (448175, 97)
Buncombe (full ArcGIS pull): (135262, 54) {'geometryType': 'esriGeometryPolygon', 'exceededTransferLimit': False, 'n_features': 135262}
Guilford (full ArcGIS pull): (223064, 84) {'geometryType': 'esriGeometryPoint', 'exceededTransferLimit': False, 'n_features': 223064}
Wake REALINFO: (472290, 87)


## 2. Which county is each dataset for, and what does a record represent?
None of the 4 files is self-labelled by filename alone (`Cama_Table_transformed.csv` gives no county), so we identify the county from the city/municipality values inside the data itself.

In [4]:
print("Cities in the CSV (loccity):")
print(sorted(meck_df["loccity"].dropna().unique()))

Cities in the CSV (loccity):
['CHARLOTTE', 'CORNELIUS', 'DAVIDSON', 'HUNTERSVILLE', 'MATTHEWS', 'MINT HILL', 'PINEVILLE', 'STALLINGS', 'UNINC']


`loccity` only ever takes values Charlotte, Cornelius, Davidson, Huntersville, Matthews, Mint Hill, Pineville, Stallings, or UNINC (unincorporated) — every one of these is a municipality of **Mecklenburg County, NC**. So despite the generic filename, this file is the Mecklenburg County CAMA table. **Grain: one row = one taxable parcel/property record** (land + building characteristics + latest sale + owner + assessed value), as of tax year 2026.

⚠️ **Gotcha caught while exploring**: `loccity` is genuinely the property's (situs) city, but `mailaddr1`, `mailaddr2`, `zipcode` and `state` are the **owner's mailing address** — e.g. rows with `state != 'NC'` show mailing zip codes from AZ/CT/SC/etc, not a Mecklenburg zip. There is no separate situs-zip column in this file.

⚠️ **Two more gotchas caught while exploring**: (1) this export uses a **comma as the decimal separator** (`gisacres`, `xcoord`, `ycoord`, `shape_Length`, ... read as text like `"0,127552"` unless `pd.read_csv(..., decimal=",")` is used — plain `float()`/arithmetic on these columns silently fails or errors without it); (2) **`xcoord`/`ycoord` are swapped** relative to the usual GIS convention — `xcoord` holds the **latitude** (~35, a valid NC latitude) and `ycoord` holds the **longitude** (~-80, a valid NC longitude), not the other way around.

In [5]:
print("Buncombe County field value:", bun_df["County"].unique())
print("Situs street sample (HouseNumber + StreetName + StreetType):")
print((bun_df["HouseNumber"] + " " + bun_df["StreetName"] + " " + bun_df["StreetType"]).head(5).tolist())
# CityName/Address/Zipcode/State are the OWNER'S MAILING address, not the property location -
# e.g. plenty of Buncombe parcels are mailed to owners in other states:
print("Owner mailing states other than NC (sample):", bun_df.loc[bun_df["State"] != "NC", "CityName"].unique()[:8])

Buncombe County field value: <StringArray>
['BUN']
Length: 1, dtype: str
Situs street sample (HouseNumber + StreetName + StreetType):


['36 ASHER LN', '14 YONDER CRK', '99999 ROBINSON COVE RD', '24 TRILLIUM CT', '99999 PATTON COVE RD']
Owner mailing states other than NC (sample): <StringArray>
[  'TRAVELERS RST',         'ANAHEIM', 'CHARLOTTESVILLE',      'GREENVILLE',
   'POMPANO BEACH',      'ROMEOVILLE',    'NEWPORT NEWS',      'ALPHARETTA']
Length: 8, dtype: str


`County == 'BUN'` (Buncombe) confirms **Buncombe County, NC** (Asheville area). **Grain: one row = one tax parcel polygon** — it comes from a cadastral/GIS layer (`geometryType: esriGeometryPolygon`), so each record is a parcel boundary plus owner, deed, land-use and assessed-value attributes. This is now the **full parcel roll** (135,262 parcels — paged with `resultOffset` until `exceededTransferLimit` was `False`, matching the service's own `returnCountOnly` total), not just the first page.

⚠️ **Gotcha caught while exploring**: `CityName`/`Address`/`Zipcode`/`State` in this file are the **owner's mailing address**, not the property's location — that's why values like `HOUSTON`, `WEST PALM BCH`, `PITTSBURGH` show up (absentee/out-of-state owners). The actual situs address is `HouseNumber + StreetName + StreetType` (+ `Township`); there is no clean situs city/zip column in this layer (the `City` field is populated only for parcels inside Black Mountain city limits, `'CBM'`, else blank).

In [6]:
print("Sample situs cities:", sorted(gui_df["SITUS_CITY"].dropna().unique())[:15])
print("Non-null rows:", gui_df["SITUS_CITY"].notna().sum(), "/", len(gui_df))

Sample situs cities: ['ARCHDALE', 'BROWN SUMMIT', 'BURLINGTON', 'CLIMAX', 'COLFAX', 'ELON', 'GIBSONVILLE', 'GREENSBORO', 'HIGH POINT', 'JAMESTOWN', 'JULIAN', 'KERNERSVILLE', 'LIBERTY', 'MCLEANSVILLE', 'OAK RIDGE']
Non-null rows: 222750 / 223064


Cities like Greensboro, Jamestown, Colfax, Elon, Randleman, Reidsville, Gibsonville place this in **Guilford County, NC** (Greensboro/High Point area). **Grain: one row = one tax parcel** — a point-geometry cadastral layer (`geometryType: esriGeometryPoint`, the parcel centroid), with building characteristics (year built, grade, heated area) plus land/building assessed values. This is now the **full parcel roll** (223,064 parcels — paged the same way as Buncombe), not just the first page.

In [7]:
print("Sample cities (PHYSICAL_CITY):", wake_df["PHYSICAL_CITY"].dropna().unique()[:10])
print(wake_df[["OWNER1","Street_Number","Street_Name","PHYSICAL_CITY","Year_Built","HEATED_AREA"]].head())

Sample cities (PHYSICAL_CITY): <StringArray>
[      'RALEIGH',       'WENDELL',          'APEX',          'CARY',
 'FUQUAY VARINA',   'WAKE FOREST',    'KNIGHTDALE',        'GARNER',
 'WILLOW SPRING', 'HOLLY SPRINGS']
Length: 10, dtype: str
                      OWNER1  Street_Number  Street_Name PHYSICAL_CITY  \
0   NORTHGATE PROPERTIES LLC           1506  WAKE FOREST       RALEIGH   
1               BONES 13 LLC           6012     TRIANGLE       RALEIGH   
2               BONES 13 LLC           6012     TRIANGLE       RALEIGH   
3  HARRISPARK PROPERTIES INC           1601  WAKE FOREST       RALEIGH   
4             MOGHADASS, INC           1831      CAPITAL       RALEIGH   

   Year_Built  HEATED_AREA  
0           0            0  
1        1979         4500  
2        1989         4008  
3        1993         8500  
4        1968         3088  


Cities such as Raleigh, Cary, Wake Forest confirm **Wake County, NC**. **Grain: one row = one *building card*, not one parcel** — `CARD_NUMBER`/`NUMBER_OF_CARDS` show a parcel (`REAL_ESTATE_ID`) can have multiple cards (e.g. multiple buildings on one lot), each with its own year built, heated area, grade, etc. This is the full county extract (no paging limit — a direct file download).

### Summary
| Dataset | County | Format | Grain | Completeness |
|---|---|---|---|---|
| `Cama_Table_transformed.csv` | Mecklenburg | CSV (`;`-delimited) | 1 row = 1 parcel/property | Full table (~448k rows) |
| `buncombe/buncombe.json` | Buncombe | ArcGIS FeatureServer JSON, polygons | 1 row = 1 parcel polygon | Full — 135,262 parcels, paged via `resultOffset` |
| `guilford/guilford.json` | Guilford | ArcGIS FeatureServer JSON, points | 1 row = 1 parcel (centroid) | Full — 223,064 parcels, paged via `resultOffset` |
| `wake/RealEstData09022026.xlsx` | Wake | Excel, sheet `REALINFO` | 1 row = 1 building card (parcel can have several) | Full extract |

The Buncombe/Guilford files come from a paged ArcGIS `query` endpoint (`outFields: "*"`, page sizes 1000/2000 in `config/config.yaml`); `src/components/data_ingestor.py` now loops on `resultOffset`, extending the requested `resultRecordCount` each call, until a page comes back shorter than requested — i.e. the whole county, not a single page.

In [8]:
print("Row counts:")
print("  Mecklenburg CSV:", len(meck_df))
print("  Buncombe JSON (full):", len(bun_df))
print("  Guilford JSON (full):", len(gui_df))
print("  Wake REALINFO:", len(wake_df))

Row counts:
  Mecklenburg CSV: 448175
  Buncombe JSON (full): 135262
  Guilford JSON (full): 223064
  Wake REALINFO: 472290


## 3. Shared columns to merge on
Each county publishes its own schema with its own column names and its own parcel-ID format (`nc_pin`/`parcelid` vs `PIN` vs `PIN_PLUS_EXTENSION`/`REID` vs `REAL_ESTATE_ID`) — **there is no literal join key shared across counties**, and that's expected: these are four disjoint sets of properties in four different counties, not four sources describing the same properties. So "merging" here means *harmonizing to a common schema and concatenating* (stacking counties into one long table), not an inner join.

The columns below carry the same real-world meaning across all four sources and are the ones to rename onto a common schema before stacking:

In [9]:
shared_schema = [
    ("county",            "loccity (+state)", "County (='BUN')", "SITUS_CITY (implied)", "PHYSICAL_CITY (implied)"),
    ("parcel_id",         "parcelid / nc_pin", "PIN", "PIN_PLUS_EXTENSION / REID", "REAL_ESTATE_ID / Parcel_Identification"),
    ("owner_name",        "ownrlstnme + ownrfrstnme", "Owner", "OWNER / OWNER1_FULLNAME", "OWNER1"),
    ("situs_street_num",  "streetnumber", "HouseNumber", "SITUS_STREET_NUMBER", "Street_Number"),
    ("situs_street_name", "streetname", "StreetName + StreetType", "SITUS_STREET + SITUS_STREET_TYPE", "Street_Name + Street_Type"),
    ("situs_city",        "loccity", "— (not present; CityName is owner mailing city)", "SITUS_CITY", "PHYSICAL_CITY"),
    ("situs_zip",         "— (not present; zipcode is owner mailing zip)", "— (not present; Zipcode is owner mailing zip)", "SITUS_ZIP", "PHYSICAL_ZIP_CODE"),
    ("sale_price",        "saleprice", "SalePrice", "PACKAGE_SALE_PRICE / LAND_SALE_PRICE", "Total_sale_Price / Land_Sale_Price"),
    ("sale_date",         "saledate", "DeedDate", "PACKAGE_SALE_DATE / DEED_DATE", "Total_Sale_Date / Deed_Date"),
    ("deed_book_page",    "deed_book + deed_page", "DeedBook + DeedPage", "DEED_BOOK_PAGE_NUMBER", "DEED_BOOK + DEED_PAGE"),
    ("land_value",        "totlandval", "LandValue", "TOTAL_LAND_VALUE_ASSESSED", "Assessed_Land_Value"),
    ("building_value",    "totalbldgval", "BuildingValue", "TOTAL_BUILDING_VALUE_ASSESSED", "Assessed_Building_Value"),
    ("total_value",       "totalvalue / totmarkval", "TotalMarketValue / TaxValue", "TOTAL_PROPERTY_VALUE", "(land_value + building_value)"),
    ("acreage",           "gisacres", "Acreage", "CALCULATED_ACREAGE / DEEDED_ACREAGE", "Deeded_Acreage"),
    ("land_use_class",    "lusecode / landuse_description", "LandUse / Class", "LAND_CLASS", "Land_classification / BILLING_CLASS / TYPE_AND_USE"),
    ("neighborhood_code", "neighborhood", "NeighborhoodCode", "NEIGHBORHOOD / APPRAISAL_NEIGHBORHOOD", "VCS"),
    ("year_built",        "yearbuilt / effyearblt", "— (not in this parcel layer)", "YEAR_BUILT", "Year_Built / Effective_year"),
    ("heated_area",       "heatedarea / finisharea", "— (not in this parcel layer)", "HEATED_AREA / MAIN_BUILDING_HEATED_AREA", "HEATED_AREA"),
    ("bed_bath",          "bedrooms, fullbath, halfbath", "— (not in this parcel layer)", "BEDROOMS, BATHROOMS, BATH_FULL, BATH_HALF", "NUM_of_Rooms, BATH, BATH_FIXTURES"),
    ("latitude_longitude","ycoord, xcoord", "_geometry.rings (polygon)", "CentroidYCoordinat, CentroidXCoordinate", "— (not provided)"),
]
schema_df = pd.DataFrame(shared_schema, columns=["common_field", "Mecklenburg (csv)", "Buncombe (json)", "Guilford (json)", "Wake (xlsx)"])
schema_df

,common_field,Mecklenburg (csv),Buncombe (json),Guilford (json),Wake (xlsx)
0,county,loccity (+state),County (='BUN'),SITUS_CITY (implied),PHYSICAL_CITY (implied)
1,parcel_id,parcelid / nc_pin,PIN,PIN_PLUS_EXTENSION / REID,REAL_ESTATE_ID / Parcel_Identification
2,owner_name,ownrlstnme + ownrfrstnme,Owner,OWNER / OWNER1_FULLNAME,OWNER1
3,situs_street_num,streetnumber,HouseNumber,SITUS_STREET_NUMBER,Street_Number
4,situs_street_name,streetname,StreetName + StreetType,SITUS_STREET + SITUS_STREET_TYPE,Street_Name + Street_Type
5,situs_city,loccity,— (not present; CityName is owner mailing city),SITUS_CITY,PHYSICAL_CITY
6,situs_zip,— (not present; zipcode is owner mailing zip),— (not present; Zipcode is owner mailing zip),SITUS_ZIP,PHYSICAL_ZIP_CODE
7,sale_price,saleprice,SalePrice,PACKAGE_SALE_PRICE / LAND_SALE_PRICE,Total_sale_Price / Land_Sale_Price
8,sale_date,saledate,DeedDate,PACKAGE_SALE_DATE / DEED_DATE,Total_Sale_Date / Deed_Date
9,deed_book_page,deed_book + deed_page,DeedBook + DeedPage,DEED_BOOK_PAGE_NUMBER,DEED_BOOK + DEED_PAGE


Notes on the mapping above:
- **Mailing address ≠ situs (property) address** in both Mecklenburg and Buncombe — their `zipcode`/`CityName`-style columns describe where the tax bill is mailed (which can be a different city, state, even a PO box), not where the property sits. Guilford and Wake avoid this ambiguity with explicit `SITUS_*`/`PHYSICAL_*` vs `OWNER_MAIL_*`/`Mailing_*` prefixes — always check for that distinction rather than trust a generic `city`/`zip`-sounding column name.
- **`county`** isn't an explicit column in most files (Guilford/Wake) — it has to be *added* per-dataset from the file's own source, since it's implied rather than stored.
- **Buncombe** is a pure cadastral/GIS layer: it has no building characteristics (no year built, heated area, bed/bath) — only parcel, owner, deed and assessed-value fields.
- **`total_value`** isn't stored directly in Wake's `REALINFO`; it's derivable as `Assessed_Land_Value + Assessed_Building_Value`.
- Parcel-ID formats are county-specific and **not** compatible with each other, so cross-county entity resolution has to go through `situs_street_num + situs_street_name + situs_zip` (address-based fuzzy match, zip only available for Guilford/Wake), not parcel ID.
- Coordinates are in different shapes (Mecklenburg: plain x/y columns; Buncombe: full polygon; Guilford: point centroid; Wake: none) — useful for mapping, but need normalizing to a single `(lat, lon)` pair per common schema.

## 4. Main idea of each dataset
- **Mecklenburg (`Cama_Table_transformed.csv`)** — the county's full CAMA (mass-appraisal) roll: every taxable parcel with its owner, latest sale price/date, land + building assessed values, and detailed building characteristics (year built, heated area, bed/bath, exterior, heat/AC, grade, garage, etc.). This is the richest single-county dataset here and the most directly usable for a house-price model on its own.
- **Buncombe (`buncombe/buncombe.json`)** — the county's full parcel/cadastral GIS layer: parcel boundaries, ownership, deed/plat references, land classification, and assessed values, but no building detail. Good for parcel geometry, zoning/land-use context, and ownership — not for structural features.
- **Guilford (`guilford/guilford.json`)** — same idea as Buncombe (a full ArcGIS parcel layer) but as point centroids, and richer: it *does* include building characteristics (year built, grade, heated area, bed/bath), so it's closer in content to the Mecklenburg CAMA table, just geometry-simplified to a centroid.
- **Wake (`wake/RealEstData09022026.xlsx`)** — a full county real-estate extract at the *building-card* grain rather than parcel grain: a parcel with multiple structures yields multiple rows (`CARD_NUMBER` of `NUMBER_OF_CARDS`), each with its own building attributes. Needs aggregating up to parcel level (`REAL_ESTATE_ID`) before it's comparable row-for-row with the other three.

**Bottom line:** these are four independent, county-specific parcel/appraisal datasets, overlapping heavily in *meaning* (owner, address, sale, assessed value, building characteristics) but not in *identifiers*. All four are now full county pulls. The natural next step is to build one loader per county that maps to the `common_field` schema above, add a `county` column, and `pd.concat` them into a single multi-county property table — Wake should be rolled up from card-level to parcel-level first.

## 5. Concatenating all four into one table
Following the `common_field` schema from section 3: each county gets its own small mapper that renames/derives its columns onto that schema (`np.nan` where a field doesn't exist for that county), then `pd.concat` stacks them into one long table with a `county` column distinguishing rows. Two things need handling first:
- **Wake is at building-card grain** — it's rolled up to one row per `REAL_ESTATE_ID` before mapping: parcel-level fields (owner, sale, land value, acreage, ...) are taken from the first card, while `heated_area` is *summed* across a parcel's cards (total living area) and `year_built` takes the *earliest* card (the original structure).
- **Buncombe has no point coordinates**, only a parcel polygon — its `latitude`/`longitude` are approximated as the mean of the polygon's exterior-ring vertices (a plain centroid approximation, not area-weighted; fine for plotting a parcel's rough location, not for precision GIS work).

In [10]:
COMMON_COLUMNS = [
    "county", "parcel_id", "owner_name", "situs_street_num", "situs_street_name", "situs_city", "situs_zip",
    "sale_price", "sale_date", "deed_book", "deed_page", "land_value", "building_value", "total_value",
    "acreage", "land_use_class", "neighborhood_code", "year_built", "heated_area", "bedrooms", "bathrooms",
    "latitude", "longitude",
]

def polygon_centroid(rings):
    """Approximate centroid: mean of the exterior ring's vertices (lon, lat)."""
    if not rings:
        return None, None
    pts = rings[0]
    lons = [p[0] for p in pts]
    lats = [p[1] for p in pts]
    return sum(lats) / len(lats), sum(lons) / len(lons)


def meck_to_common(df):
    out = pd.DataFrame({
        "county": "Mecklenburg",
        "parcel_id": df["parcelid"],
        "owner_name": (df["ownrlstnme"].fillna("") + " " + df["ownrfrstnme"].fillna("")).str.strip(),
        "situs_street_num": df["streetnumber"],
        "situs_street_name": df["streetname"],
        "situs_city": df["loccity"],
        "situs_zip": pd.NA,  # only the owner's mailing zip is present, not situs
        "sale_price": df["saleprice"],
        "sale_date": pd.to_datetime(df["saledate"], format="%d/%m/%y %H:%M", errors="coerce"),
        "deed_book": df["deed_book"],
        "deed_page": df["deed_page"],
        "land_value": df["totlandval"],
        "building_value": df["totalbldgval"],
        "total_value": df["totalvalue"],
        "acreage": df["gisacres"],
        "land_use_class": df["landuse_description"],
        "neighborhood_code": df["neighborhood"],
        "year_built": df["yearbuilt"],
        "heated_area": df["heatedarea"],
        "bedrooms": df["bedrooms"],
        "bathrooms": df["fullbath"] + 0.5 * df["halfbath"],
        "latitude": df["xcoord"],   # NB: this county's export has x/y swapped vs. the usual lon/lat convention
        "longitude": df["ycoord"],  # (xcoord holds ~35 = latitude, ycoord holds ~-80 = longitude for Mecklenburg)
    })
    return out[COMMON_COLUMNS]


def bun_to_common(df):
    lat, lon = zip(*(polygon_centroid(g["rings"]) if g else (None, None) for g in df["_geometry"]))
    out = pd.DataFrame({
        "county": "Buncombe",
        "parcel_id": df["PIN"],
        "owner_name": df["Owner"],
        "situs_street_num": df["HouseNumber"],
        "situs_street_name": (df["StreetName"].fillna("") + " " + df["StreetType"].fillna("")).str.strip(),
        "situs_city": pd.NA,  # CityName is the owner's mailing city, not situs
        "situs_zip": pd.NA,   # Zipcode is the owner's mailing zip, not situs
        "sale_price": df["SalePrice"],
        "sale_date": pd.to_datetime(df["DeedDate"], format="%Y%m%d", errors="coerce"),  # deed date, used as sale-date proxy
        "deed_book": df["DeedBook"],
        "deed_page": df["DeedPage"],
        "land_value": df["LandValue"],
        "building_value": df["BuildingValue"],
        "total_value": df["TotalMarketValue"],
        "acreage": df["Acreage"],
        "land_use_class": df["LandUse"].replace("", pd.NA).fillna(df["Class"]),
        "neighborhood_code": df["NeighborhoodCode"],
        "year_built": pd.NA,     # not captured in this parcel/GIS layer
        "heated_area": pd.NA,    # not captured in this parcel/GIS layer
        "bedrooms": pd.NA,       # not captured in this parcel/GIS layer
        "bathrooms": pd.NA,      # not captured in this parcel/GIS layer
        "latitude": lat,
        "longitude": lon,
    })
    return out[COMMON_COLUMNS]


def gui_to_common(df):
    out = pd.DataFrame({
        "county": "Guilford",
        "parcel_id": df["PIN_PLUS_EXTENSION"],
        "owner_name": df["OWNER1_FULLNAME"].fillna(df["OWNER"]),
        "situs_street_num": df["SITUS_STREET_NUMBER"],
        "situs_street_name": (df["SITUS_STREET"].fillna("") + " " + df["SITUS_STREET_TYPE"].fillna("")).str.strip(),
        "situs_city": df["SITUS_CITY"],
        "situs_zip": df["SITUS_ZIP"],
        "sale_price": df["PACKAGE_SALE_PRICE"],
        "sale_date": pd.to_datetime(df["PACKAGE_SALE_DATE"], unit="ms", errors="coerce"),
        "deed_book": df["DEED_BOOK_PAGE_NUMBER"].str.split("-").str[0],
        "deed_page": df["DEED_BOOK_PAGE_NUMBER"].str.split("-").str[1],
        "land_value": df["TOTAL_LAND_VALUE_ASSESSED"],
        "building_value": df["TOTAL_BUILDING_VALUE_ASSESSED"],
        "total_value": df["TOTAL_PROPERTY_VALUE"],
        "acreage": df["CALCULATED_ACREAGE"],
        "land_use_class": df["LAND_CLASS"],
        "neighborhood_code": df["NEIGHBORHOOD"],
        "year_built": df["YEAR_BUILT"],
        "heated_area": df["HEATED_AREA"],
        "bedrooms": df["BEDROOMS"],
        "bathrooms": df["BATHROOMS"],
        "latitude": df["CentroidYCoordinat"],
        "longitude": df["CentroidXCoordinate"],
    })
    return out[COMMON_COLUMNS]


def wake_rollup_to_parcel(df):
    """Wake's REALINFO is one row per building card; collapse to one row per REAL_ESTATE_ID (parcel)."""
    first = df.sort_values("CARD_NUMBER").groupby("REAL_ESTATE_ID", as_index=False).first()
    agg = df.groupby("REAL_ESTATE_ID").agg(
        heated_area_total=("HEATED_AREA", "sum"),
        year_built_earliest=("Year_Built", "min"),
    ).reset_index()
    return first.merge(agg, on="REAL_ESTATE_ID")


def wake_to_common(df):
    parcels = wake_rollup_to_parcel(df)
    out = pd.DataFrame({
        "county": "Wake",
        "parcel_id": parcels["REAL_ESTATE_ID"],
        "owner_name": parcels["OWNER1"],
        "situs_street_num": parcels["Street_Number"],
        "situs_street_name": (parcels["Street_Name"].fillna("") + " " + parcels["Street_Type"].fillna("")).str.strip(),
        "situs_city": parcels["PHYSICAL_CITY"],
        "situs_zip": parcels["PHYSICAL_ZIP_CODE"],
        "sale_price": parcels["Total_sale_Price"],
        "sale_date": pd.to_datetime(parcels["Total_Sale_Date"], errors="coerce"),
        "deed_book": parcels["DEED_BOOK"],
        "deed_page": parcels["DEED_PAGE"],
        "land_value": parcels["Assessed_Land_Value"],
        "building_value": parcels["Assessed_Building_Value"],
        "total_value": parcels["Assessed_Land_Value"] + parcels["Assessed_Building_Value"],
        "acreage": parcels["Deeded_Acreage"],
        "land_use_class": parcels["Land_classification"],
        "neighborhood_code": parcels["VCS"],
        "year_built": parcels["year_built_earliest"],
        "heated_area": parcels["heated_area_total"],
        "bedrooms": pd.NA,   # not captured (only room/bath codes, not clean counts)
        "bathrooms": pd.NA,  # BATH is a rating code here, not a fixture count
        "latitude": pd.NA,   # no coordinates in this extract
        "longitude": pd.NA,
    })
    return out[COMMON_COLUMNS]


properties_df = pd.concat(
    [meck_to_common(meck_df), bun_to_common(bun_df), gui_to_common(gui_df), wake_to_common(wake_df)],
    ignore_index=True,
)
properties_df.shape

(1262796, 23)

In [11]:
print(properties_df["county"].value_counts())
print()
print("Missing (%) by common field, per county:")
(properties_df.isna().groupby(properties_df["county"]).mean() * 100).round(1).T

county
Wake           456295
Mecklenburg    448175
Guilford       223064
Buncombe       135262
Name: count, dtype: int64

Missing (%) by common field, per county:


county,Buncombe,Guilford,Mecklenburg,Wake
county,0.0,0.0,0.0,0.0
parcel_id,0.0,0.8,0.5,0.0
owner_name,0.0,0.0,0.0,0.0
situs_street_num,0.0,0.0,4.4,0.0
situs_street_name,0.0,0.0,0.0,0.0
situs_city,100.0,0.1,0.0,0.6
situs_zip,100.0,1.5,100.0,1.1
sale_price,0.0,28.9,15.5,0.0
sale_date,0.0,28.9,0.3,17.8
deed_book,0.0,0.0,1.5,0.2


In [12]:
properties_df.head()

,county,parcel_id,owner_name,situs_street_num,situs_street_name,situs_city,situs_zip,sale_price,sale_date,deed_book,...,total_value,acreage,land_use_class,neighborhood_code,year_built,heated_area,bedrooms,bathrooms,latitude,longitude
0,Mecklenburg,4310652.0,RINKU INC,8800.0,MICHAW CT,CHARLOTTE,<NA>,NaN,2024-12-10,39259.0,...,307000.0,0.127552,SINGLE FAMILY RESIDENTIAL,E316,2004.0,1664.0,3.0,2.0,35.319288,-80.804261
1,Mecklenburg,540388.0,RANDLE CHRISTOPHER R,19343.0,COURTNEY JAE LN,CORNELIUS,<NA>,155000.0,2006-12-29,21585.0,...,386500.0,0.127990,SINGLE FAMILY RESIDENTIAL,A525,2006.0,1767.0,3.0,2.5,35.466465,-80.857693
2,Mecklenburg,19333473.0,SIMPSON SHEILA C,12911.0,PHILLIPS RD,MATTHEWS,<NA>,0.0,2025-11-13,40001.0,...,516800.0,0.312093,SINGLE FAMILY RESIDENTIAL,T324,2003.0,3267.0,3.0,2.5,35.132624,-80.68831
3,Mecklenburg,3133336.0,PHOMSOPHA SOMPHAK,614.0,STILLGREEN LN,CHARLOTTE,<NA>,117000.0,2008-08-06,24024.0,...,311200.0,0.155219,SINGLE FAMILY RESIDENTIAL,D118,2004.0,1600.0,3.0,2.5,35.298793,-80.937647
4,Mecklenburg,20130145.0,BLOOD OF THE LAMB DELIVERANCE CHURCH,2735.0,ARROWOOD RD,CHARLOTTE,<NA>,350000.0,2012-01-24,27050.0,...,2390600.0,2.919375,CHURCH,OF03,2019.0,7544.0,NaN,NaN,35.147445,-80.933137


The combined table has **one row per property** (parcel, except where a county only tracks land — see the per-column missing-% breakdown above for exactly which fields each county doesn't provide): Mecklenburg and Guilford are the most complete (full building characteristics); Buncombe has none of the building fields (`year_built`, `heated_area`, `bedrooms`, `bathrooms`); Wake is missing coordinates and clean bedroom/bathroom counts. `sale_price`/`sale_date` and `situs_city`/`situs_zip` gaps mostly trace back to the mailing-vs-situs and page/format caveats noted in sections 2 and 3.

Saved to `data/processed/properties_combined.csv` for reuse without re-parsing all four raw sources.

In [13]:
PROCESSED_PATH = ROOT_PATH + "/" + cfg["paths"]["processed_data"]
import os
os.makedirs(PROCESSED_PATH, exist_ok=True)
properties_df.to_csv(PROCESSED_PATH + "properties_combined.csv", index=False)
print("wrote", len(properties_df), "rows to", PROCESSED_PATH + "properties_combined.csv")

wrote 1262796 rows to /Users/a70411/Let-me-have-a-house//data/processed/properties_combined.csv
